In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
import os
sys.path.append('/rhome/sawale/indus_traning/mlm-fine-tuning/mlm')

from preprocess_data import preprocess_dataset_with_static_masking
import json

config_path = "../config_new_data.json"
with open(config_path, "r") as file:
    config = json.load(file)

data_src = "local"
n_rows = 16

tokenized_ds, tokenizer, data_collator = preprocess_dataset_with_static_masking(
        config.get("input"),
        data_src,
        n_rows,
    )

# masker = KeywordMasking(tokenizer, "text")
# lm_dataset = masker.mask_keywords(tokenized_ds)

/rhome/sawale/indus_traning/mlm-fine-tuning/xenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map (num_proc=4): 100%|██████████| 12/12 [00:00<00:00, 42.40 examples/s]
num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
Map (num_proc=2): 100%|██████████| 2/2 [00:00<00:00,  8.74 examples/s]
num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
Gen Prob Matrix (num_proc=8): 100%|██████████| 13/13 [00:01<00:00,  6.54 examples/s]
num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.
Gen Prob Matrix (num_proc=3): 100%|██████████| 3/3 [00:00<00:00,  3.38 examples/s]
num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
Static Masking (num_proc=8): 100%|██████████| 13/13 [00:00<00:00, 37.63 examples/s]
num_proc must be <= 3. Reducing num_proc to 3

In [11]:

text = "Name: hello my name is Nish"

tds = tokenizer(text, return_offsets_mapping=True,)
tds


{'input_ids': [50281, 2402, 27, 23120, 619, 1416, 310, 427, 763, 50282], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'offset_mapping': [(0, 0), (0, 4), (4, 5), (5, 11), (11, 14), (14, 19), (19, 22), (22, 24), (24, 27), (0, 0)]}

In [12]:
[text[s:t] for s,t in [(0, 0), (0, 4), (4, 5), (5, 11), (11, 14), (14, 19), (19, 22), (22, 24), (24, 27), (0, 0)]]

['', 'Name', ':', ' hello', ' my', ' name', ' is', ' N', 'ish', '']

In [85]:
from itertools import chain

flat_list = list(chain(*lm_dataset["train"]["labels"]))

flat_att = list(chain(*lm_dataset["train"]["attention_mask"]))
n = sum(flat_att)

m = sum([1 for i in flat_list if i != -100])

print(m, n, m/n)

718 9114 0.07877989905639675


In [86]:
for i, j in zip(lm_dataset["train"]["input_ids"][5], lm_dataset["train"]["labels"][5]):
    print(i, j, tokenizer.decode(i), tokenizer.decode(j) if j != -100 else "NOT")

50281 -100 [CLS] NOT
1992 -100 To NOT
50284 3057 [MASK]  contact
253 -100  the NOT
7565 -100  Earth NOT
50284 6875 [MASK]  Science
50284 12664 [MASK]  Week
37630 -100  organizers NOT
27 -100 : NOT
187 -100 
 NOT
13425 -100 General NOT
50284 34989 [MASK]  inquiries
8692 -100  info NOT
33 -100 @ NOT
29500 -100 earth NOT
29409 -100 sci NOT
11151 -100 week NOT
15 -100 . NOT
2061 -100 org NOT
187 -100 
 NOT
29150 -100 Director NOT
273 -100  of NOT
35758 10286 mounted  Education
285 -100  and NOT
6282 -100  Out NOT
21943 -100 reach NOT
50284 12824 [MASK]  Edward
6625 -100  Rob NOT
70 -100 e NOT
777 -100 ck NOT
10038 -100  ec NOT
287 -100 ro NOT
28298 -100 beck NOT
33 -100 @ NOT
40463 -100 americ NOT
912 -100 ange NOT
5829 -100 osc NOT
7545 -100 iences NOT
15 -100 . NOT
2061 -100 org NOT
818 -100  7 NOT
2941 -100 03 NOT
14 -100 - NOT
24880 -100 379 NOT
14 -100 - NOT
1348 -100 24 NOT
1438 -100 80 NOT
50284 1021 [MASK]  ext
15 -100 . NOT
22752 -100  245 NOT
187 -100 
 NOT
1992 -100 To NOT
1340 

In [92]:
def find_max_iou_edit_distance(word, keywords):
        """Finds max IoU using edit distance as the difference measure."""
        max_iou = 0.0
        for keyword in keywords:
            dist = edit_distance(word, keyword)
            union = len(word) + len(keyword) - dist
            iou= (union - dist) / union if union > 0 else 0.0
            max_iou = max(max_iou, iou)
        return max_iou

word = " "
keywords = ["hello", "."]

find_max_iou_edit_distance(word, keywords)

0.0

In [164]:
# Example text
text = ["This is an example sentence. hah", "you are my evething"]

# Tokenize with both return_overflowing_tokens and return_offsets_mapping
encoding = tokenizer(
    text,
    max_length= 6,
    truncation=True,
    padding="max_length",
    return_overflowing_tokens=True,
    return_length=True,
    return_offsets_mapping=True
)

print(encoding)

{'input_ids': [[50281, 1552, 310, 271, 1650, 50282], [50281, 6197, 15, 419, 73, 50282], [50281, 5658, 403, 619, 612, 50282], [50281, 1526, 50282, 50283, 50283, 50283]], 'attention_mask': [[1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1], [1, 1, 1, 0, 0, 0]], 'offset_mapping': [[(0, 0), (0, 4), (4, 7), (7, 10), (10, 18), (0, 0)], [(0, 0), (18, 27), (27, 28), (28, 31), (31, 32), (0, 0)], [(0, 0), (0, 3), (3, 7), (7, 10), (10, 13), (0, 0)], [(0, 0), (13, 19), (0, 0), (0, 0), (0, 0), (0, 0)]], 'length': [6, 6, 6, 6], 'overflow_to_sample_mapping': [0, 0, 1, 1]}


In [ ]:
text = ["a", "b", "c", "d"]

mappings = [0,1,1,2,3,3]

output = ["a", "b", "b", "c", "d", "d"]
# how to make this mappenings to output

